# Week 6 Hands-On Lab — Prompting and Evaluating a Video World Model

**ESP3201 · formative hands-on lab.** Generation needs a free-tier Colab **GPU (T4) runtime**. If no GPU is available, use the **precomputed rollout-bank** fallback — you still do the full evaluation on real frames.

A video-generation model is a **world model**: a learned simulator of how a scene evolves. You will vary **prompting**, **input controls** (seed, frames, guidance), and **evaluation criteria**, and measure how each affects the quality, consistency, and *usefulness* of the rollout.

> **Report only frames your run actually produced** (or the provided rollout-bank clips). No projected results.

**Attribution.** Evaluation dimensions follow **VBench** (Huang et al., 2023); generation follows the **diffusers** text-to-video tutorials. Lab code is original to ESP3201.

## Setup

In [ ]:
import os
from IPython.display import display, Video

os.system('pip install -q numpy pillow diffusers imageio imageio-ffmpeg')

# --- Week 6 lab core, embedded directly (no repo clone) -----------------------
# Canonical source, kept in sync by hand: starter/video_eval.py in the course
# repo. Cloning a support module from Colab is fragile -- if a session already
# ran once before an update landed, the clone silently no-ops onto the stale
# cached copy instead of fetching the fix. Pasting the needed code directly
# here removes that whole failure class. (The precomputed rollout_bank/ clips
# used below are binary image data, not code -- they're base64-embedded in a
# separate cell right after this one, for the same reason.)

from typing import Dict, List, Optional

import numpy as np


Frame = np.ndarray  # HxWx3 uint8 or float


def _to_float(frames: List[Frame]) -> np.ndarray:
    arr = np.stack([np.asarray(f, dtype=np.float32) for f in frames])
    if arr.max() > 1.5:
        arr = arr / 255.0
    return arr  # (T, H, W, C) in [0, 1]


def mean_abs_diff(frames: List[Frame]) -> float:
    """Mean absolute per-pixel change between consecutive frames (motion)."""
    arr = _to_float(frames)
    if arr.shape[0] < 2:
        return 0.0
    diffs = np.abs(arr[1:] - arr[:-1])
    return float(diffs.mean())


def temporal_consistency(frames: List[Frame]) -> float:
    """1 - motion. High = visually stable across frames."""
    return float(1.0 - mean_abs_diff(frames))


def flicker(frames: List[Frame]) -> float:
    """Std over time of global mean brightness (global temporal instability)."""
    arr = _to_float(frames)
    per_frame = arr.reshape(arr.shape[0], -1).mean(axis=1)
    return float(per_frame.std())


def brightness_drift(frames: List[Frame]) -> float:
    """Max deviation of global brightness from the first frame."""
    arr = _to_float(frames)
    per_frame = arr.reshape(arr.shape[0], -1).mean(axis=1)
    return float(np.abs(per_frame - per_frame[0]).max())


def score_clip(frames: List[Frame], label: str = "") -> Dict[str, object]:
    """Objective metrics plus slots for the human-judged criteria."""
    return {
        "label": label,
        "n_frames": len(frames),
        "temporal_consistency": round(temporal_consistency(frames), 4),
        "motion_magnitude": round(mean_abs_diff(frames), 4),
        "flicker": round(flicker(frames), 4),
        "brightness_drift": round(brightness_drift(frames), 4),
        # human-judged in the worksheet (1-5):
        "prompt_adherence": None,
        "object_identity_stability": None,
        "physical_plausibility": None,
    }


# --------------------------------------------------------------------------- #
# Loaders / fallbacks
# --------------------------------------------------------------------------- #
def load_frames_from_dir(path: str) -> List[Frame]:
    """Load PNG/JPG frames (sorted by name) from a directory.

    Use this for the precomputed rollout-bank fallback when no GPU is available:
    point it at labs/week06_video_world_model_study/rollout_bank/<clip>/.
    """
    import glob
    import os
    from PIL import Image
    files = sorted(glob.glob(os.path.join(path, "*.png")) +
                   glob.glob(os.path.join(path, "*.jpg")))
    return [np.asarray(Image.open(f).convert("RGB")) for f in files]


def synth_clip(kind: str, n: int = 16, size: int = 64, seed: int = 0) -> List[Frame]:
    """Synthetic clips to validate the metrics offline (no model needed):
      'static'  -> a still image (consistency ~1, motion ~0)
      'moving'  -> a square translating smoothly (some motion, low flicker)
      'noise'   -> random frames (high motion, high flicker)
    """
    rng = np.random.default_rng(seed)
    frames = []
    if kind == "static":
        base = rng.random((size, size, 3)).astype(np.float32)
        frames = [base.copy() for _ in range(n)]
    elif kind == "moving":
        for t in range(n):
            f = np.full((size, size, 3), 0.1, dtype=np.float32)
            x = int((size - 12) * t / max(1, n - 1))
            f[size // 2 - 6:size // 2 + 6, x:x + 12, :] = [0.9, 0.2, 0.2]
            frames.append(f)
    elif kind == "noise":
        frames = [rng.random((size, size, 3)).astype(np.float32) for _ in range(n)]
    else:
        raise ValueError(f"unknown kind '{kind}'")
    return frames


def frames_from_diffusers(output) -> List[Frame]:
    """Adapt a diffusers text-to-video output to a list of RGB frames.

    Handles both formats seen across pipelines: `output.frames[0]` as a list of
    PIL Images, or (e.g. zeroscope/ModelScope TextToVideoSDPipeline) as numpy
    arrays in [0, 1]. Verified against cerspense/zeroscope_v2_576w.
    """
    frames = output.frames[0]
    out = []
    for f in frames:
        if hasattr(f, "convert"):           # PIL image
            out.append(np.asarray(f.convert("RGB")))
        else:                                # numpy array (H, W, 3)
            out.append(np.asarray(f))
    return out


def show_frames(frames: List[Frame], n: int = 8):
    """Lay out up to n evenly-spaced frames as one strip image, each panel
    captioned with its frame index.

    Returns a PIL Image, which Jupyter/Colab renders inline -- one image per
    clip that's easy to screenshot or right-click-save straight into a report,
    with the frame numbers already on it so a critique can cite "frame 12"
    and point at exactly the panel that shows it. Handles both uint8 [0,255]
    and float [0,1] frame arrays.
    """
    from PIL import Image, ImageDraw
    idxs = np.linspace(0, len(frames) - 1, min(n, len(frames))).round().astype(int)
    thumbs = []
    for i in idxs:
        f = np.asarray(frames[int(i)])
        if f.dtype != np.uint8:
            f = (np.clip(f, 0.0, 1.0) * 255).round().astype(np.uint8)
        thumbs.append((int(i), Image.fromarray(f)))
    w, h = thumbs[0][1].size
    label_h = 18
    strip = Image.new("RGB", (w * len(thumbs), h + label_h), "white")
    draw = ImageDraw.Draw(strip)
    for k, (i, im) in enumerate(thumbs):
        strip.paste(im, (k * w, 0))
        draw.text((k * w + 4, h + 2), f"frame {i}", fill=(0, 0, 0))
    return strip


def compare_frames(named_frames: "Dict[str, Frame]"):
    """Lay out named single frames side by side, each captioned with its label.

    Use this for side-by-side comparisons that are NOT a time sequence --
    e.g. original vs. VAE reconstruction, or a latent-interpolation sweep --
    as opposed to show_frames(), which captions panels by frame index.
    """
    from PIL import Image, ImageDraw
    thumbs = []
    for label, f in named_frames.items():
        arr = np.asarray(f)
        if arr.dtype != np.uint8:
            arr = (np.clip(arr, 0.0, 1.0) * 255).round().astype(np.uint8)
        thumbs.append((str(label), Image.fromarray(arr)))
    w, h = thumbs[0][1].size
    label_h = 20
    strip = Image.new("RGB", (w * len(thumbs), h + label_h), "white")
    draw = ImageDraw.Draw(strip)
    for k, (label, im) in enumerate(thumbs):
        strip.paste(im, (k * w, 0))
        draw.text((k * w + 4, h + 2), label, fill=(0, 0, 0))
    return strip


def save_video(frames: List[Frame], fps: int = 8, path: Optional[str] = None) -> str:
    """Encode frames to a playable .mp4 and return its file path.

    Feed the returned path to IPython.display.Video(path, embed=True) in a
    notebook cell to get an actual play/pause/scrub player alongside the
    frame strip -- embed=True inlines the video as base64 so it survives
    Colab's remote-VM file serving instead of trying to link a local path.

    Frames are normalized to uint8 PIL images before handing off to
    diffusers' export_to_video(): that function's own numpy-array path
    assumes float [0,1] input and multiplies by 255 unconditionally, which
    silently corrupts already-uint8 [0,255] frames (overflow wraparound).
    Going through PIL images sidesteps that entirely.
    """
    from PIL import Image
    from diffusers.utils import export_to_video
    import tempfile
    pil_frames = []
    for f in frames:
        arr = np.asarray(f)
        if arr.dtype != np.uint8:
            arr = (np.clip(arr, 0.0, 1.0) * 255).round().astype(np.uint8)
        pil_frames.append(Image.fromarray(arr))
    if path is None:
        path = tempfile.NamedTemporaryFile(suffix=".mp4", delete=False).name
    return export_to_video(pil_frames, path, fps=fps)

def show_clip(frames, fps=8, n=8, display_width=360):
    """Display a clip both ways: a playable video, and a labeled frame strip
    you can screenshot straight into a report. The video needs diffusers +
    imageio + imageio-ffmpeg (installed above) but NOT torch or a GPU --
    this works on Track B too.

    display_width forces a legible player size regardless of the clip's native
    resolution -- the warm-up synthetic clips are only 64x64px, at which the
    browser's play/pause control bar renders too small to see or click."""
    try:
        display(Video(save_video(frames, fps=fps), embed=True, width=display_width))
    except Exception as e:
        print('Video encode failed (frame strip below still works):', repr(e))
    display(show_frames(frames, n=n))

print('video_eval loaded (embedded in this notebook -- no repo clone needed).')

In [ ]:
# --- Precomputed rollout-bank frames, embedded directly (no repo clone) ------
# Same rationale as the code above: a scoped clone of esp3201-public was fragile
# (stale cache, wrong cwd, or the data simply not existing there once the public
# mirror stopped carrying non-notebook files). These are the same two compact
# 8-frame/384x213 clips as rollout_bank/ in the course repo, base64-embedded so
# this cell is the only source of truth -- nothing to go stale or go missing.
import base64
import io

_ROLLOUT_BANK_B64 = {
    "clip01_arm_pick": [
        "...",
    ],
    "clip02_ball_drop": [
        "...",
    ],
}

ROLLOUT_BANK_CLIPS = list(_ROLLOUT_BANK_B64)


def load_bank_frames(clip_name: str) -> List[Frame]:
    """Decode one embedded rollout-bank clip back into a list of uint8 RGB frames."""
    from PIL import Image
    frames = []
    for b64 in _ROLLOUT_BANK_B64[clip_name]:
        img = Image.open(io.BytesIO(base64.b64decode(b64))).convert("RGB")
        frames.append(np.asarray(img))
    return frames

print(f"rollout bank loaded ({len(ROLLOUT_BANK_CLIPS)} clips, embedded -- no repo clone needed).")


In [ ]:
# Diagram helper -- run once. Small matplotlib helpers used throughout this
# notebook to give each concept a picture alongside its text description.
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import FancyBboxPatch, Arc

_STATE_STYLE = {
    "new":      dict(facecolor="#e9f1fd", edgecolor="#2563eb", linewidth=2.2, fontweight="bold", fontcolor="#1d4ed8"),
    "carried":  dict(facecolor="#f1f5f9", edgecolor="#94a3b8", linewidth=1.3, fontweight="normal", fontcolor="#475569"),
    "evidence": dict(facecolor="#e6f5ef", edgecolor="#047857", linewidth=2.0, fontweight="bold", fontcolor="#036c4e"),
    "risk":     dict(facecolor="#fdecea", edgecolor="#e74c3c", linewidth=1.8, fontweight="bold", fontcolor="#c0392b"),
}
_BOX_W, _BOX_H, _GAP, _ROW_H = 2.15, 0.92, 0.55, 1.35

def _draw_row(ax, y, stages, arrows=True):
    x = 0.0
    centers = []
    for st in stages:
        style = _STATE_STYLE[st.get("state", "carried")]
        box = FancyBboxPatch((x, y), _BOX_W, _BOX_H, boxstyle="round,pad=0.02,rounding_size=0.09",
                              linewidth=style["linewidth"], edgecolor=style["edgecolor"],
                              facecolor=style["facecolor"], zorder=2)
        ax.add_patch(box)
        cx, cy = x + _BOX_W / 2, y + _BOX_H / 2
        sub = st.get("sub")
        ax.text(cx, cy + (0.15 if sub else 0), st["label"], ha="center", va="center",
                fontsize=10.3, fontweight=style["fontweight"], color=style["fontcolor"], zorder=3)
        if sub:
            ax.text(cx, cy - 0.22, sub, ha="center", va="center", fontsize=8.4,
                     color=style["fontcolor"], zorder=3)
        centers.append((cx, cy))
        x += _BOX_W + _GAP
    if arrows:
        for (x1, y1), (x2, y2) in zip(centers, centers[1:]):
            ax.annotate("", xy=(x2 - _BOX_W / 2 - 0.04, y2), xytext=(x1 + _BOX_W / 2 + 0.04, y1),
                        arrowprops=dict(arrowstyle="-|>", color="#64748b", lw=1.5), zorder=1)
    return x - _GAP

def show_pipeline_diagram(title, rows, note=None, figsize=None):
    # rows: list of {"label": optional row caption, "stages": [{"label","sub","state"}], "arrows": bool}
    n_rows = len(rows)
    has_labels = any(r.get("label") for r in rows)
    total_h = n_rows * _ROW_H
    fig_h = 0.55 + total_h + (0.35 if note else 0)
    fig, ax = plt.subplots(figsize=figsize or (11.8, fig_h))
    ax.axis("off")
    max_x = 0
    y_cursor = total_h
    for row in rows:
        y_cursor -= _ROW_H
        y = y_cursor
        rx = _draw_row(ax, y, row["stages"], arrows=row.get("arrows", True))
        max_x = max(max_x, rx)
        if row.get("label"):
            ax.text(-0.2, y + _BOX_H / 2, row["label"], ha="right", va="center",
                     fontsize=9.3, fontweight="bold", color="#334155")
    ax.set_xlim(-2.6 if has_labels else -0.3, max_x + 0.4)
    ax.set_ylim(-0.35, total_h + 0.35)
    ax.set_title(title, fontsize=12.5, fontweight="bold", color="#1a1a2e", loc="left", pad=8)
    if note:
        fig.text(0.02, 0.01, note, fontsize=8.8, color="#64748b", style="italic")
    # Shows and closes the figure rather than returning it: an earlier version returned
    # fig as well as calling plt.show(), so Jupyter/Colab rendered it a second time as
    # the cell's auto-displayed result whenever a diagram call was a cell's only statement.
    plt.tight_layout()
    plt.show()
    plt.close(fig)

def show_guidance_diagram():
    fig, axes = plt.subplots(1, 3, figsize=(12.5, 4.6))
    scales = [1.0, 9.0, 15.0]
    uncond = np.array([0.35, 0.18])
    cond = np.array([0.55, 0.85])
    direction = cond - uncond
    for ax, scale in zip(axes, scales):
        guided = uncond + scale * direction
        lim = max(1.3, np.linalg.norm(guided) * 1.2)
        ax.set_xlim(-lim * 0.25, lim); ax.set_ylim(-lim * 0.15, lim)
        ax.axhline(0, color="#e2e8f0", lw=1); ax.axvline(0, color="#e2e8f0", lw=1)

        ax.annotate("", xy=uncond, xytext=(0, 0),
                    arrowprops=dict(arrowstyle="-|>", color="#94a3b8", lw=2.2))
        ax.text(uncond[0] - lim * 0.02, uncond[1] - lim * 0.09, "uncond", color="#64748b",
                fontsize=9.5, fontweight="bold", ha="right")

        same_as_cond = abs(scale - 1.0) < 1e-9
        color = "#047857" if scale <= 9.0 else "#e74c3c"

        ax.annotate("", xy=cond, xytext=(0, 0),
                    arrowprops=dict(arrowstyle="-|>", color="#2563eb", lw=2.2))
        if not same_as_cond:
            ax.text(cond[0] + lim * 0.02, cond[1] - lim * 0.02, "cond (prompt)", color="#1d4ed8",
                    fontsize=9.5, fontweight="bold", ha="left", va="top")
            ax.annotate("", xy=guided, xytext=(0, 0),
                        arrowprops=dict(arrowstyle="-|>", color=color, lw=2.8))
            ax.text(guided[0] + lim * 0.03, guided[1] + lim * 0.02, "guided", color=color,
                    fontsize=10, fontweight="bold", ha="left")
        else:
            ax.text(cond[0] + lim * 0.03, cond[1] + lim * 0.02,
                    "guided = cond\n(scale 1.0 applies\nno extra push)",
                    color=color, fontsize=9.5, fontweight="bold", ha="left")

        tag = "under-guided" if scale == 1.0 else ("pinned default" if scale == 9.0 else "over-guided\n(risk of artifacts)")
        ax.set_title(f"guidance_scale = {scale}\n{tag}", fontsize=10.5, fontweight="bold")
        ax.set_aspect("equal"); ax.set_xticks([]); ax.set_yticks([])
        for s in ax.spines.values():
            s.set_visible(False)
    fig.suptitle("Classifier-Free Guidance: guided = uncond + scale x (cond - uncond)", fontsize=12.5, fontweight="bold")
    # Shows and closes the figure rather than returning it: an earlier version returned
    # fig as well as calling plt.show(), so Jupyter/Colab rendered it a second time as
    # the cell's auto-displayed result whenever a diagram call was a cell's only statement.
    plt.tight_layout()
    plt.show()
    plt.close(fig)

def show_slerp_diagram():
    fig, ax = plt.subplots(figsize=(7.5, 7))
    theta = np.linspace(0, 2 * np.pi, 200)
    ax.plot(np.cos(theta), np.sin(theta), color="#cbd5e1", lw=1.5, zorder=1)
    angle_a, angle_b = 20, 110
    A = np.array([np.cos(np.radians(angle_a)), np.sin(np.radians(angle_a))])
    B = np.array([np.cos(np.radians(angle_b)), np.sin(np.radians(angle_b))])
    lerp_mid = (A + B) / 2
    mid_angle = (angle_a + angle_b) / 2
    slerp_mid = np.array([np.cos(np.radians(mid_angle)), np.sin(np.radians(mid_angle))])

    ax.plot([0, A[0]], [0, A[1]], color="#2563eb", lw=2)
    ax.plot([0, B[0]], [0, B[1]], color="#2563eb", lw=2)
    ax.scatter(*A, color="#2563eb", s=70, zorder=5)
    ax.text(A[0] * 1.14, A[1] * 1.14, "A", fontsize=13, fontweight="bold", color="#1d4ed8")
    ax.scatter(*B, color="#2563eb", s=70, zorder=5)
    ax.text(B[0] * 1.14, B[1] * 1.14, "B", fontsize=13, fontweight="bold", color="#1d4ed8")

    ax.plot([A[0], B[0]], [A[1], B[1]], color="#e74c3c", lw=2.2, linestyle="--", zorder=4)
    ax.scatter(*lerp_mid, color="#e74c3c", s=80, zorder=6)
    ax.annotate("lerp(0.5)\nshorter norm\n(off the circle)", lerp_mid,
                xytext=(lerp_mid[0] - 0.05, lerp_mid[1] - 0.55),
                fontsize=9, color="#c0392b", ha="center", fontweight="bold",
                arrowprops=dict(arrowstyle="-", color="#c0392b", lw=1))

    arc = Arc((0, 0), 2, 2, angle=0, theta1=angle_a, theta2=angle_b, color="#047857", lw=2.6, zorder=4)
    ax.add_patch(arc)
    ax.scatter(*slerp_mid, color="#047857", s=80, zorder=6)
    ax.annotate("slerp(0.5)\nsame norm\n(stays on the circle)", slerp_mid,
                xytext=(slerp_mid[0] + 0.55, slerp_mid[1] + 0.35),
                fontsize=9, color="#036c4e", ha="center", fontweight="bold",
                arrowprops=dict(arrowstyle="-", color="#036c4e", lw=1))

    ax.plot([0, 0], [0, 0], color="#e74c3c", lw=2.2, linestyle="--", label="lerp path (chord) -- leaves the circle")
    ax.plot([0, 0], [0, 0], color="#047857", lw=2.6, label="slerp path (arc) -- stays on the circle")
    ax.legend(loc="lower left", fontsize=9, frameon=False)

    ax.set_aspect("equal"); ax.set_xlim(-1.5, 1.5); ax.set_ylim(-1.5, 1.5)
    ax.set_xticks([]); ax.set_yticks([])
    for s in ax.spines.values():
        s.set_visible(False)
    ax.set_title("slerp vs lerp: two noise vectors on the unit-norm circle", fontsize=12.5, fontweight="bold")
    fig.text(0.5, 0.02, "The circle is the sphere every training noise sample lives on. "
                        "slerp walks along it; lerp cuts through the inside.",
              fontsize=9, color="#64748b", style="italic", ha="center")
    # Shows and closes the figure rather than returning it: an earlier version returned
    # fig as well as calling plt.show(), so Jupyter/Colab rendered it a second time as
    # the cell's auto-displayed result whenever a diagram call was a cell's only statement.
    plt.tight_layout()
    plt.show()
    plt.close(fig)

print('Diagram helpers ready: show_pipeline_diagram, show_guidance_diagram, show_slerp_diagram')

## Warm-up — what the metrics mean (no model needed)

Three synthetic clips show how the objective metrics behave: a static clip, a smoothly moving object, and pure noise.

In [ ]:
for kind in ('static', 'moving', 'noise'):
    frames = synth_clip(kind)
    print(score_clip(frames, label=kind))
    show_clip(frames, fps=8)

**What `video_eval` is actually computing.** Every metric above is a plain, offline comparison between adjacent frames — no model, no learned scorer, nothing that understands what an "arm" or a "cube" is:

- `temporal_consistency` = 1 − mean absolute pixel-value change between consecutive frames. High = visually stable.
- `motion_magnitude` is that same mean absolute change — the complement of consistency (`consistency = 1 − motion`).
- `flicker` is the standard deviation, over time, of each frame's overall brightness — a proxy for *global* instability (e.g. the whole frame suddenly darkening or flashing), not local object motion.
- `brightness_drift` is how far a frame's brightness ever wanders from frame 0, at its worst point.

Look at the strip above alongside the numbers: **static** is consistency ≈ 1 / motion ≈ 0 (frozen); **moving** has real but small motion with near-zero flicker (smooth, coherent); **noise** has high motion *and* high flicker (incoherent — nothing to track frame to frame). That is the whole diagnostic: high consistency with near-zero motion means *nothing is happening*; high flicker means *global incoherence*. A useful world-model rollout needs motion that is consistent, not frozen and not chaotic — and because these are pixel statistics only, they cannot tell you whether the motion is *physically correct*, which is why the worksheet also asks for your own human judgment (prompt adherence, identity stability, physical plausibility).

## Generate clips with a real video model (Colab GPU)

Pinned to **`cerspense/zeroscope_v2_576w`**, **verified on a free T4** (fp16 + CPU offload, measured ~5.8 GB peak; see `pinned_models.md`). Swap `MODEL_ID` and re-check memory if you like.

> **Time budget (important).** On a free T4 each clip takes **roughly 1–3 minutes** (much slower than the lab's reference GPU). The default below runs **3 experiments**, which is enough to see the controls matter; only add more if your session has time. Generate one clip first to gauge your runtime before launching the grid.

In [ ]:
# Run on a GPU runtime.
os.system('pip install -q diffusers transformers accelerate torch')
import torch
from diffusers import DiffusionPipeline

MODEL_ID = 'cerspense/zeroscope_v2_576w'   # VERIFIED on free T4 (~5.8 GB fp16+offload)
pipe = DiffusionPipeline.from_pretrained(MODEL_ID, torch_dtype=torch.float16)
pipe.enable_model_cpu_offload()
try:
    pipe.enable_vae_slicing()
except Exception:
    pass

def generate(prompt, seed=0, num_frames=24, num_inference_steps=25,
             guidance_scale=9.0, negative_prompt=None):
    g = torch.Generator(device='cuda').manual_seed(seed)
    out = pipe(prompt, num_frames=num_frames, num_inference_steps=num_inference_steps,
               guidance_scale=guidance_scale, negative_prompt=negative_prompt,
               height=320, width=576, generator=g)
    return frames_from_diffusers(out)   # zeroscope returns numpy frames; adapter handles it

print('peak VRAM after load (GB):', round(torch.cuda.max_memory_allocated()/1e9, 2))

In [ ]:
show_pipeline_diagram(
    "How Part 1 Generation Works -- Where Each Control Acts",
    rows=[{"stages": [
        {"label": "PROMPT + SEED", "sub": "text, negative_prompt, seed", "state": "carried"},
        {"label": "STARTING NOISE", "sub": "random latent tensor", "state": "carried"},
        {"label": "U-NET DENOISER", "sub": "num_inference_steps x guidance_scale", "state": "new"},
        {"label": "VAE DECODER", "sub": "per-frame, latent -> pixels", "state": "new"},
        {"label": "VIDEO FRAMES", "sub": "what score_clip() measures", "state": "evidence"},
    ]}],
    note="prompt/negative_prompt and seed set the starting point; steps and guidance_scale control the U-Net's denoising; only the VAE decoder turns latents into pixels.",
)

### Controlled experiments (3 by default)

Each changes **one factor** so you can attribute the effect:

1. **terse** vs **detailed** prompt — does detail + motion verbs help?
2. **detailed + negative prompt** — does a fuller negative list clean it up further?

**A note on prompt wording for this small model.** `zeroscope_v2_576w` is a ~1.5B-parameter model -- much smaller than the video models the lecture covers. Naive extra description (piling on adjectives) can actually *hurt* it: the added tokens compete for attention and the model sometimes renders several weak, scattered copies of the object instead of one coherent one. The **detailed** prompt below is deliberately structured like a camera/production brief (subject, action, setting, then a short list of quality terms) rather than just more adjectives -- that ordering is what keeps it coherent. Keep this in mind if you edit the prompts yourself.

Once these run, the worksheet invites you to add a **seed** change and a **guidance** change *if time allows*.

In [ ]:
experiments = {
  'terse':         dict(prompt='a robot arm picking up a cube'),
  'detailed':      dict(prompt='cinematic shot of an industrial robotic arm slowly and precisely '
                               'grasping a red cube on a wooden table, smooth continuous motion, '
                               'sharp focus, natural studio lighting, high detail, professional footage'),
  'detailed+neg':  dict(prompt='cinematic shot of an industrial robotic arm slowly and precisely '
                               'grasping a red cube on a wooden table, smooth continuous motion, '
                               'sharp focus, natural studio lighting, high detail, professional footage',
                        negative_prompt='blurry, distorted, flickering, low quality, low resolution, '
                                        'jittery, warped, deformed, extra limbs, disfigured, static '
                                        'noise, glitch, oversaturated, grainy'),
}
scores = []
for label, kw in experiments.items():
    frames = generate(**kw)
    scores.append(score_clip(frames, label=label))
    print(scores[-1])
    # A playable video plus a labeled frame strip you can screenshot/
    # right-click-save straight into your report worksheet -- cite the
    # frame numbers the strip shows when you write up what changed.
    show_clip(frames, fps=8)

# Kept under its own name: later cells (e.g. the rollout-bank fallback) reuse
# the name `frames` for their own loops, which would otherwise silently steal
# Exercise 1's "prefer your own live generation" check.
last_generated_frames = frames

In [ ]:
# Bar chart -- every objective metric for every experiment you just ran, at a glance.
metrics = ['temporal_consistency', 'motion_magnitude', 'flicker', 'brightness_drift']
labels = [s['label'] for s in scores]
colors = ['#2563eb', '#d97706', '#047857', '#e74c3c', '#94a3b8']

fig, axes = plt.subplots(1, len(metrics), figsize=(15, 3.6))
for ax, metric in zip(axes, metrics):
    values = [s[metric] for s in scores]
    bars = ax.bar(labels, values, color=colors[:len(labels)])
    ax.set_title(metric, fontsize=10.5, fontweight='bold')
    ax.tick_params(axis='x', rotation=25, labelsize=8.5)
    for b, v in zip(bars, values):
        ax.text(b.get_x() + b.get_width() / 2, v, f'{v:.3f}', ha='center', va='bottom', fontsize=8)
    ax.margins(y=0.15)
fig.suptitle('Controlled experiments -- objective metrics at a glance', fontweight='bold')
plt.tight_layout()
plt.show()

### Optional extra controls (only if your session has time)

```python
# same prompt, different seed -> how stable is the model?
print(score_clip(generate('a robot arm picking up a cube', seed=1), label='seedB'))
# higher guidance -> tighter prompt adherence, but can over-sharpen / flicker
print(score_clip(generate('a robot arm picking up a cube', guidance_scale=14.0), label='high_guidance'))
```

## No GPU? Use the precomputed rollout bank

Evaluate real generated frames committed under `rollout_bank/` without generating them yourself.

In [ ]:
for name in ROLLOUT_BANK_CLIPS:
    # Named bank_frames, not frames -- Part 1's `frames` (your live GPU
    # generation) must survive this cell untouched, since Exercise 1 below
    # prefers it when it exists. Reusing the name `frames` here would
    # silently make Exercise 1 always fall back to the rollout bank, even
    # after a real generation ran.
    bank_frames = load_bank_frames(name)
    print(score_clip(bank_frames, label=name))
    show_clip(bank_frames, fps=4)  # rollout-bank clips are the compact 8-frame sample


## Part 2 — Opening the Box: How the Model Actually Works

Part 1 treated the model as a black box and scored what came out. Part 2 pries the box open: you'll look at the VAE that compresses each frame, and at what the diffusion sampler's own controls (step count, guidance) and its starting noise actually do.

**Honest architecture note.** The lecture's "causal 3D VAE" describes newer-generation video models. `zeroscope_v2_576w` is an older-generation ModelScope-style pipeline: its VAE is the same **per-frame 2D image VAE** used in Stable Diffusion (no temporal awareness at all) — all temporal structure lives in the U-Net's 3D convolutions, not the VAE. That contrast is itself worth reporting: it means this pinned model's VAE round-trip result tells you about *per-frame* compression only, not about how it handles *time*.

**Compute/track note.** Exercise 1 (VAE round-trip) runs on CPU with no full pipeline needed, so it works for both tracks. Exercises 2–4 need the full GPU pipeline from Part 1's generation cell (Track A only) — Track B students note this in the worksheet instead of running them.

### Exercise 1 — VAE round-trip

Encode one real frame through the VAE, decode it straight back, and compare. This is the cheapest possible experiment here (a couple of forward passes, no iterative denoising) and it makes "the VAE is a lossy compressor" into something you can see: blur, lost fine texture, color drift.

In [ ]:
show_pipeline_diagram(
    "Exercise 1 -- The VAE Round-Trip",
    rows=[{"stages": [
        {"label": "IMAGE", "sub": "real frame, e.g. 384x213x3", "state": "carried"},
        {"label": "VAE ENCODER", "sub": "vae.encode()", "state": "new"},
        {"label": "LATENT", "sub": "compressed, e.g. 4x27x48", "state": "risk"},
        {"label": "VAE DECODER", "sub": "vae.decode()", "state": "new"},
        {"label": "RECONSTRUCTION", "sub": "blur / color drift vs original", "state": "evidence"},
    ]}],
    note="No diffusion here -- two forward passes only. Whatever changes between IMAGE and RECONSTRUCTION is purely the VAE's compression loss.",
)

In [ ]:
import torch
import numpy as np
import glob
from PIL import Image, ImageDraw

def _as_uint8_frame(frame):
    """Normalize a frame to uint8 [0,255] regardless of source convention --
    the rollout bank loads real uint8 PNGs via PIL, but a live zeroscope
    generation returns float32 arrays in [0,1] (see frames_from_diffusers'
    own docstring). Feeding a float [0,1] frame straight into the '/127.5 -
    1.0' VAE normalization below silently mis-scales it (everything collapses
    toward -1) instead of erroring, and PIL's Image.fromarray() outright
    rejects a float32 RGB array later on -- this is the one normalization
    point that has to handle both."""
    arr = np.asarray(frame)
    if arr.dtype != np.uint8:
        arr = (np.clip(arr, 0.0, 1.0) * 255).round().astype(np.uint8)
    return arr

# Works whether or not you ran the GPU generation cells above.
try:
    vae = pipe.vae.to('cuda' if torch.cuda.is_available() else 'cpu')
    # ^ pipe.vae's own .device attribute is unreliable under enable_model_cpu_offload():
    # it can claim 'cuda' while the weights are actually still sitting on CPU (the
    # offload hook only fires on the pipeline's forward(), not on a raw .encode()/.decode()
    # call). Forcing it explicitly here avoids a "tensors on different devices" error.
    print('Using the VAE from the already-loaded pipeline.')
except NameError:
    from diffusers import AutoencoderKL
    vae = AutoencoderKL.from_pretrained('cerspense/zeroscope_v2_576w', subfolder='vae')
    print('No pipeline loaded -- loaded just the VAE component (CPU, no GPU needed for this exercise).')

def vae_roundtrip(frame_uint8):
    """Encode one real frame through the VAE and decode it straight back.

    Crops to a multiple of 8 first: this VAE downsamples/upsamples by exactly
    8x, so an input size that isn't divisible by 8 (e.g. the rollout-bank's
    compact 213x384 sample) decodes back at a slightly different size than it
    started at, which breaks a direct pixel-difference comparison.
    """
    device = next(vae.parameters()).device
    frame_uint8 = _as_uint8_frame(frame_uint8)
    h, w = frame_uint8.shape[:2]
    h8, w8 = h - (h % 8), w - (w % 8)
    frame_uint8 = frame_uint8[:h8, :w8]
    x = torch.from_numpy(frame_uint8.astype(np.float32) / 127.5 - 1.0)  # -> [-1, 1]
    x = x.permute(2, 0, 1).unsqueeze(0).to(vae.dtype).to(device)
    with torch.no_grad():
        latent = vae.encode(x).latent_dist.sample() * vae.config.scaling_factor
        recon = vae.decode(latent / vae.config.scaling_factor).sample
    recon = ((recon.clamp(-1, 1) + 1) * 127.5).squeeze(0).permute(1, 2, 0).cpu().numpy().astype(np.uint8)
    return recon, latent, frame_uint8  # cropped source too, for a fair comparison

def gather_three_source_frames():
    """Three genuinely different real frames -- from your own generation if you ran
    one above (three different time points in the same clip), else combined across
    the rollout bank's clips (it ships 2 clips x 8 frames, not 3 separate clips, so
    a frame-0-only pick across clips wouldn't reach 3 distinct images)."""
    try:
        fr = last_generated_frames
        idxs = (0, len(fr) // 2, len(fr) - 1)
        return [f'frame {i}' for i in idxs], [fr[i] for i in idxs]
    except NameError:
        pool = []
        for name in ROLLOUT_BANK_CLIPS:
            fr = load_bank_frames(name)
            pool.append((f'{name} frame 0', fr[0]))
            pool.append((f'{name} frame {len(fr)//2}', fr[len(fr) // 2]))
        return [p[0] for p in pool[:3]], [p[1] for p in pool[:3]]

src_labels, src_frames_raw = gather_three_source_frames()

roundtrip_results = []
for label, src in zip(src_labels, src_frames_raw):
    recon, latent, cropped = vae_roundtrip(src)
    compression_ratio = (cropped.shape[0] * cropped.shape[1] * cropped.shape[2]) / latent.numel()
    err = round(mean_abs_diff([cropped, recon]), 4)
    roundtrip_results.append(dict(label=label, shape=cropped.shape, latent_shape=tuple(latent.shape),
                                   compression_ratio=round(compression_ratio, 1), error=err))
    print(f'[{label}] original {cropped.shape} -> latent {tuple(latent.shape)}  '
          f'compression: {compression_ratio:.1f}x  recon error: {err}')
    display(compare_frames({f'{label}: original': cropped, f'{label}: VAE reconstruction': recon}))

# --- Perturb a region of the encoded latent, then decode -- what does the VAE
# "invent" for a patch of the image whose true latent content is destroyed? ---
def vae_perturb_region(frame_uint8, frac=0.35, seed=0):
    """Encode, overwrite a centered square patch of the LATENT (not the pixels)
    with fresh random noise scaled to the latent's own std, then decode. This
    probes what the decoder does with an out-of-distribution latent patch --
    unlike the round-trip above, which shows loss on an UNCHANGED latent."""
    device = next(vae.parameters()).device
    frame_uint8 = _as_uint8_frame(frame_uint8)
    h, w = frame_uint8.shape[:2]
    h8, w8 = h - (h % 8), w - (w % 8)
    frame_uint8 = frame_uint8[:h8, :w8]
    x = torch.from_numpy(frame_uint8.astype(np.float32) / 127.5 - 1.0)
    x = x.permute(2, 0, 1).unsqueeze(0).to(vae.dtype).to(device)
    with torch.no_grad():
        latent = vae.encode(x).latent_dist.sample() * vae.config.scaling_factor
        perturbed = latent.clone()
        lh, lw = perturbed.shape[-2:]
        ph, pw = int(lh * frac), int(lw * frac)
        y0, x0 = (lh - ph) // 2, (lw - pw) // 2
        g = torch.Generator(device=device).manual_seed(seed)
        patch = perturbed[..., y0:y0 + ph, x0:x0 + pw]
        noise = torch.randn(patch.shape, generator=g, device=device, dtype=perturbed.dtype)
        perturbed[..., y0:y0 + ph, x0:x0 + pw] = noise * perturbed.std()
        recon_perturbed = vae.decode(perturbed / vae.config.scaling_factor).sample
    recon_perturbed = ((recon_perturbed.clamp(-1, 1) + 1) * 127.5).squeeze(0).permute(1, 2, 0).cpu().numpy().astype(np.uint8)
    scale = h8 // lh  # this VAE's fixed 8x downsample factor
    box = (x0 * scale, y0 * scale, (x0 + pw) * scale, (y0 + ph) * scale)
    return recon_perturbed, box, frame_uint8

recon_pert, box, src_for_pert = vae_perturb_region(src_frames_raw[0])
marked_original = Image.fromarray(src_for_pert).copy()
ImageDraw.Draw(marked_original).rectangle(box, outline=(255, 0, 0), width=3)
print(f'Perturbed a {box[2]-box[0]}x{box[3]-box[1]}px region (in pixel space) of the latent, then decoded.')
display(compare_frames({
    f'{src_labels[0]}: original (perturbed region marked)': np.asarray(marked_original),
    f'{src_labels[0]}: reconstruction with perturbed latent': recon_pert,
}))

### Exercise 2 — Diffusion step-count sweep (GPU / Track A)

The pinned setting is 25 inference steps. Fewer steps means less iterative denoising — same starting noise, same prompt, just less time spent refining it. This costs one more generation per value (~1-3 min each on a free T4).

In [ ]:
show_pipeline_diagram(
    "Exercise 2 -- Fewer vs More Denoising Steps",
    rows=[
        {"label": "5 steps", "stages": [
            {"label": "NOISE", "state": "carried"},
            {"label": "few refinements", "state": "risk"},
            {"label": "UNDER-DENOISED", "sub": "coarse, less resolved", "state": "risk"},
        ]},
        {"label": "25 steps", "stages": [
            {"label": "NOISE", "state": "carried"},
            {"label": "many refinements", "state": "new"},
            {"label": "REFINED FRAME", "sub": "pinned default", "state": "evidence"},
        ]},
    ],
    note="Same starting noise, same prompt -- num_inference_steps is the only thing that changes between rows.",
)

In [ ]:
# GPU required (Track A) -- needs `generate` from the Part 1 pipeline cell.
step_sweep = {}
for steps in (5, 15, 25):
    frames_i = generate('a robot arm picking up a cube', num_inference_steps=steps)
    step_sweep[steps] = score_clip(frames_i, label=f'{steps} steps')
    print(step_sweep[steps])
    show_clip(frames_i, fps=8)

### Exercise 3 — Guidance-scale sweep (GPU / Track A)

Guidance scale controls how hard the model is pushed to match the text prompt versus its own unconditional prior. Low guidance drifts from the prompt; very high guidance can over-sharpen or flicker (the pinned default is 9.0).

In [ ]:
show_guidance_diagram()

In [ ]:
# GPU required (Track A) -- needs `generate` from the Part 1 pipeline cell.
guidance_sweep = {}
for g in (1.0, 9.0, 15.0):
    frames_i = generate('a robot arm picking up a cube', guidance_scale=g)
    guidance_sweep[g] = score_clip(frames_i, label=f'guidance={g}')
    print(guidance_sweep[g])
    show_clip(frames_i, fps=8)

### Exercise 4 — Latent-space interpolation (GPU / Track A)

Every generation starts from a random noise tensor (the "latent" the diffusion sampler denoises step by step). Here we build two different starting-noise tensors from two seeds and walk between them with **slerp** (spherical interpolation), running the **full** sampler from each interpolated starting point with the same prompt.

**Why slerp, not a plain average.** Two independent random Gaussian noise tensors are very close to orthogonal in this many dimensions, so their plain average (lerp) has a *smaller norm* than either one — verified here: norm(A) ≈ 526, norm(B) ≈ 526, but norm(lerp(A,B,0.5)) ≈ 372. Decoding that shrunk, off-manifold point produces a flat, nearly featureless frame — not a meaningful "in-between" scene, just a degenerate one. Slerp keeps the norm constant along the path (≈526 throughout), which is why it's the standard recipe for latent-space walks: it stays on the sphere the model was actually trained to denoise from.

**What this cell actually generates and shows** — four full, independent clips, not one comparison grid: **latent A** (t=0), **latent B** (t=1), the **slerp midpoint**, and — so the degeneracy above is something you *see*, not just read — the **lerp midpoint** too. All four use the same prompt and step/guidance settings; only the starting latent differs. A second, optional cell below repeats all four with the **detailed** prompt from Part 1, since the terse prompt alone doesn't tell you whether better prompting can buy back any of what interpolation costs.

In [ ]:
show_slerp_diagram()

In [ ]:
# GPU required (Track A). Verified against the pinned checkpoint. Learns the
# exact latent tensor shape (rather than assuming it) via a cheap 1-step probe.
try:
    _shape_probe = {}

    def _grab_shape(step, timestep, latents):
        # NOTE: this pipeline (TextToVideoSDPipeline) is on the OLDER diffusers
        # callback API -- callback(step, timestep, latents), not the newer
        # callback_on_step_end(pipe, step, timestep, callback_kwargs). If you
        # swap MODEL_ID for a newer pipeline, check its __call__ signature
        # (inspect.signature(pipe.__call__)) before assuming either style.
        _shape_probe['shape'] = latents.shape
        _shape_probe['dtype'] = latents.dtype
        _shape_probe['device'] = latents.device

    # Must match height/width used below, or this silently probes the
    # pipeline's *default* resolution instead of the pinned 320x576.
    _ = pipe('a robot arm picking up a cube', num_frames=24, num_inference_steps=1, height=320, width=576,
             callback=_grab_shape, callback_steps=1)
    shape, dtype, device = _shape_probe['shape'], _shape_probe['dtype'], _shape_probe['device']
    print('learned latent shape:', tuple(shape))

    def make_init_latents(seed):
        g = torch.Generator(device=device).manual_seed(seed)
        return torch.randn(shape, generator=g, device=device, dtype=dtype)

    def slerp(a, b, t):
        """Spherical interpolation -- keeps the norm constant (see markdown above)."""
        a_f, b_f = a.flatten().float(), b.flatten().float()
        a_n, b_n = a_f / a_f.norm(), b_f / b_f.norm()
        omega = torch.acos((a_n * b_n).sum().clamp(-1, 1))
        if omega.abs() < 1e-4:
            return (1 - t) * a + t * b
        so = torch.sin(omega)
        out = (torch.sin((1 - t) * omega) / so) * a_f + (torch.sin(t * omega) / so) * b_f
        return out.reshape(a.shape).to(a.dtype)

    def lerp(a, b, t):
        """Plain linear interpolation -- the 'wrong tool' this exercise contrasts
        against slerp (see the norm collapse in the markdown above)."""
        return ((1 - t) * a.float() + t * b.float()).reshape(a.shape).to(a.dtype)

    def run_interpolation_grid(prompt, label, seed_a=0, seed_b=7, steps=25, guidance=9.0):
        """Generate and show all four conditions for one prompt: latent A, latent
        B, the slerp midpoint, and the lerp midpoint. Each is a full, independent
        run of the sampler from that starting latent."""
        latent_A = make_init_latents(seed_a)
        latent_B = make_init_latents(seed_b)
        lerp_mid = lerp(latent_A, latent_B, 0.5)
        slerp_mid = slerp(latent_A, latent_B, 0.5)
        print(f'[{label}] norm A: {round(latent_A.float().norm().item(), 1)}'
              f'  norm B: {round(latent_B.float().norm().item(), 1)}'
              f'  norm lerp(0.5): {round(lerp_mid.float().norm().item(), 1)}'
              f'  norm slerp(0.5): {round(slerp_mid.float().norm().item(), 1)}')

        conditions = {
            'latent A (t=0)': latent_A,
            'latent B (t=1)': latent_B,
            'slerp midpoint (t=0.5)': slerp_mid,
            'lerp midpoint (t=0.5) -- the wrong tool': lerp_mid,
        }
        results = {}
        for cond_label, latents in conditions.items():
            out = pipe(prompt, num_frames=24, num_inference_steps=steps,
                       guidance_scale=guidance, height=320, width=576, latents=latents)
            fr = frames_from_diffusers(out)
            results[cond_label] = score_clip(fr, label=f'{label}: {cond_label}')
            print(results[cond_label])
            show_clip(fr, fps=8)
        return results

    terse_interp_results = run_interpolation_grid('a robot arm picking up a cube', label='terse')
except Exception as e:
    print('This cell needs the live GPU pipeline from Part 1. If your diffusers version '
          'differs from the one this was verified against, the callback API may have '
          'changed again -- check inspect.signature(pipe.__call__) before assuming.')
    print('Error (report this verbatim in your write-up, do not guess what it would show):')
    print(repr(e))

#### Optional: repeat with a detailed prompt (only if your session has time)

The terse prompt above is already where the model's quality has the least headroom -- interpolated latents make that worse. Repeating the same four-condition grid with the tuned **detailed** prompt from Part 1 shows whether better prompting also buys back some of what interpolation costs, or whether the noise-space degeneracy (especially lerp's) is a separate problem prompting can't fix.

In [ ]:
# Optional -- only if your session has time (4 more ~1-3 min generations on a free T4).
try:
    DETAILED_PROMPT = ('cinematic shot of an industrial robotic arm slowly and precisely '
                       'grasping a red cube on a wooden table, smooth continuous motion, '
                       'sharp focus, natural studio lighting, high detail, professional footage')
    detailed_interp_results = run_interpolation_grid(DETAILED_PROMPT, label='detailed')
except NameError as e:
    print('This needs run_interpolation_grid from the required cell above to have succeeded first.')
    print('Error (report this verbatim, do not guess what it would show):')
    print(repr(e))

## Worksheet (your deliverable)

> **Submit one PDF, 5 pages maximum.** The cap is hard — it includes every table, chart, frame-strip, and screenshot. **Condense:** prefer a chart or a single frame-strip that *compares* your conditions over long tables or many images. Every part below that says "embed" or "report" needs the actual image/frame-strip from your own run, not a description of what you expect it would show. Finish with the required **"How to improve this assignment"** section (last section).

### 1. Controls x criteria grid

Record objective metrics + your 1-5 human judgments for each experiment you ran (see the bar-chart cell for the objective metrics at a glance):

| Experiment | Temporal consistency | Motion | Flicker | Prompt adherence (1-5) | Identity stability (1-5) | Physical plausibility (1-5) |
|------------|----------------------|--------|---------|------------------------|--------------------------|-----------------------------|
| terse | | | | | | |
| detailed | | | | | | |
| detailed+neg | | | | | | |
| *(optional)* seedB | | | | | | |
| *(optional)* high_guidance | | | | | | |

### 2. Which controls mattered?

- Which single control most improved a criterion you care about, with the numbers/judgments that show it?
- Where did the metrics and your eyes **disagree** (e.g. high consistency but the object morphed)? What does that say about trusting any single metric?

### 3. World-model-as-predictor

- At what `num_frames` / horizon did coherence break for your model?
- `Would you trust this model's rollout inside a planning loop? For what horizon, and why?`

### 4. Part 2 — Opening the Box (required)

**4a. VAE round-trip.** For each of your 3 source images, report: original frame shape, latent shape, compression ratio, mean-abs-diff reconstruction error, and embed the original/reconstruction comparison image. In 1-2 sentences: what specifically got lost (fine texture? color? edges?) and does that match "the VAE is a per-frame 2D compressor with no notion of time" from the intro note? Then, for the perturbed-latent part: embed the marked-original/perturbed-reconstruction comparison and describe what the decoder produced in the perturbed region -- a plausible fill-in, a smeared blur, or something unrelated to the surrounding content?

**4b. Diffusion step-count sweep (Track A).** Table: steps (5/15/25) x the four objective metrics. At what step count did quality visibly plateau? Was there a step count where the clip was clearly still under-denoised (describe what that looked like)?

**4c. Guidance-scale sweep (Track A).** Table: guidance (1.0/9.0/15.0) x the four objective metrics + your prompt-adherence judgment (1-5). Did high guidance visibly over-sharpen or flicker, as the intro predicted? Cite the frame(s) where you saw it, if any.

**4d. Latent-space interpolation (Track A).** Report the four norms printed (A, B, lerp midpoint, slerp midpoint) and embed frame strips for all four conditions (latent A, latent B, slerp midpoint, lerp midpoint). Is the slerp midpoint a plausible "in-between" scene, or does it look unrelated to both endpoints? What does the lerp midpoint look like, and does that match the norm-collapse prediction? If you ran the optional detailed-prompt repeat, does better prompting change what you see in the interpolated clips, or is the degeneracy a separate problem? **If the cell errored for you:** paste the exact error and stop there — do not describe an expected result you did not observe.

**Track B (no GPU):** do 4a only (it runs on CPU). For 4b-4d, write 2-3 sentences on what you would predict WOULD happen for each control based on the lecture's diffusion-mechanics material, explicitly labeled as an untested prediction, not an observation.

## How to improve this assignment (required, ungraded)

*Required for a complete submission; it carries no marks.* In 3–5 sentences: what was unclear, too easy, too hard, or missing here? Name the **one change** that would make this a better learning exercise or a fairer test of the skill — a different probe, a harder map / scene / model, a metric that would have caught something this one missed, or a clearer instruction. Be specific; "it was fine" is not useful feedback.

## AI-Agent Usage Disclosure

State:

- which tools you used
- what they helped produce
- what you verified or rewrote yourself
- one specific thing you did not trust without checking